# Gaussian Elimination in BRNS

This notebook shows the **internal** symbolic Gaussian elimination workflow used in `acg_brns` to reduce reaction systems before building residuals and Jacobians for BRNS.

In practical BRNS models, Gaussian elimination is mainly useful for three reasons:

1. **System reduction**: detect inert components and conservation relations
2. **Smaller Jacobians**: fewer active equations can reduce Newton solver cost
3. **Robust symbolic workflow**: build residuals and Jacobians from explicit reaction-rate expressions

## What you should take away
- You can switch Gaussian elimination on or off in the YAML-file 

## 1. Import Required Libraries

Import SymPy for symbolic math and the BRNS Gaussian elimination entry point:

In [7]:
from sympy import symbols, S, simplify, Matrix
from acg_brns.gaussian_elimination import run_gaussian_elimination

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Simple System: 3 Components, 2 Reactions

Let us start with a **simplified didactic system**:
- **3 components**: C1, C2, C3
- **2 reactions**: r1, r2 (used as **symbolic placeholders**)

**Stoichiometry:**
- C1: consumed by r1 and r2 → dC1/dt = -r1 - r2
- C2: produced by r1, consumed by r2 → dC2/dt = r1 - r2
- C3: produced by r1 and r2 → dC3/dt = 2*r1 + 3*r2

**Note:** In this simple example we use r1 and r2 as symbols, not as concrete rate expressions.
For real systems, see Section 4, where reaction rates are defined explicitly (for example `r1 = k1*C1*C2`).

In [8]:
# Define symbolic variables
C1, C2, C3 = symbols('C1:4')
C1_old, C2_old, C3_old = symbols('C1_old C2_old C3_old')
r1, r2 = symbols('r1 r2')
delt = symbols('delt', positive=True, real=True)

# Define stoichiometric equations (linear in r1, r2)
# NOTE: This is a simplified example using symbolic rates r1, r2
# For real systems, you would define r1 = k1*C1*C2, etc. (see Section 4)
equations = [
    -r1 - r2,       # dC1/dt = -r1 - r2
    r1 - r2,        # dC2/dt = r1 - r2
    2*r1 + 3*r2     # dC3/dt = 2*r1 + 3*r2
]

print("Stoichiometric Equations (linear in symbolic r1, r2):")
for i, eq in enumerate(equations, 1):
    print(f"  Component {i}: {eq}")

Stoichiometric Equations:
  Component 1: -r1 - r2
  Component 2: r1 - r2
  Component 3: 2*r1 + 3*r2


### Run Gaussian Elimination Pipeline (p4-p10)

In [9]:
# Run full Gaussian elimination pipeline
result = run_gaussian_elimination(
    equations=equations,
    variables=[C1, C2, C3],
    reactions=[r1, r2],
    variables_old=[C1_old, C2_old, C3_old],
    delt=delt,
    verbose=False
)

print("\n" + "="*70)
print("GAUSSIAN ELIMINATION RESULTS (p4-p10)")
print("="*70)


GAUSSIAN ELIMINATION RESULTS (p4-p10)


### Residual Equations with Implicit Euler Time Discretization

In [10]:
print("\nResidual Equations F(C) with implicit Euler time discretization:")
print("F(C) = R(C) - C/delt + C_old/delt\n")
for i, func in enumerate(result['func'], 1):
    print(f"F{i}(C) = {func}")


Residual Equations F(C) with implicit Euler time discretization:
F(C) = R(C) - C/delt + C_old/delt

F1(C) = (-C3 + C3_old)/delt
F2(C) = (-C2 + C2_old + C3/2 - C3_old/2 + delt*r2)/delt
F3(C) = 2*(-C3 + C3_old)/(5*delt)


### Jacobian Matrix (3×3)

pd[i,j] = ∂F_i/∂C_j (Partial derivatives of residuals w.r.t. concentrations)

In [11]:
print("\nJacobian Matrix J[i,j] = ∂F_i/∂C_j:")
jacobian = result['jacobian']
print(f"\nShape: {jacobian.shape}\n")

for i in range(3):
    for j in range(3):
        print(f"  J[{i+1},{j+1}] = {jacobian[i,j]}")


Jacobian Matrix J[i,j] = ∂F_i/∂C_j:

Shape: (3, 3)

  J[1,1] = 0
  J[1,2] = 0
  J[1,3] = -1/delt
  J[2,1] = 0
  J[2,2] = -1/delt
  J[2,3] = 1/(2*delt)
  J[3,1] = 0
  J[3,2] = 0
  J[3,3] = -2/(5*delt)


### Stoichiometric Matrix M

This matrix contains the transformed stoichiometric coefficients after Gaussian elimination.

In [12]:
print("\nStoichiometric Matrix M (3×2):")
print("\nM =")
for i in range(3):
    row = [str(result['matrixM'][i,j]) for j in range(2)]
    print(f"  [{row[0]:>10s} | {row[1]:>10s}]")

print(f"\nPivot rows used in elimination: {result['pivot_rows']}")


Stoichiometric Matrix M (3×2):

M =
  [         0 |          0]
  [         0 |          1]
  [         0 |          0]

Pivot rows used in elimination: [0, 1]


## 3. System with Inert Component

Let us extend the system to **4 components**, where **C4 is inert** (it does not participate in reactions).

In [13]:
# 4-component system where C4 is inert
C1, C2, C3, C4 = symbols('C1:5')
C1_old, C2_old, C3_old, C4_old = symbols('C1_old C2_old C3_old C4_old')
r1, r2 = symbols('r1 r2')

equations_inert = [
    -r1 - r2,       # C1 reactive
    r1 - r2,        # C2 reactive
    2*r1 + 3*r2,    # C3 reactive
    S(0)            # C4 inert (no reactions)
]

print("4-Component System with Inert Component:")
for i, eq in enumerate(equations_inert, 1):
    status = "(INERT)" if i == 4 else ""
    print(f"  Component {i}: {eq} {status}")

4-Component System with Inert Component:
  Component 1: -r1 - r2 
  Component 2: r1 - r2 
  Component 3: 2*r1 + 3*r2 
  Component 4: 0 (INERT)


### Run with Inert Component Handling

In [14]:
# Run Gaussian elimination with inert_components parameter
result_inert = run_gaussian_elimination(
    equations=equations_inert,
    variables=[C1, C2, C3, C4],
    reactions=[r1, r2],
    variables_old=[C1_old, C2_old, C3_old, C4_old],
    delt=delt,
    inert_components={3},  # C4 is at index 3
    verbose=False
)

print("\nInert Component Handling:")
print(f"  F4(C) = {result_inert['func'][3]}")
print(f"\n(Inert components have only time discretization term,")
print(f" no reaction contributions)")


Inert Component Handling:
  F4(C) = (-C4 + C4_old)/delt

(Inert components have only time discretization term,
 no reaction contributions)


## 4. Robust Usage with Explicit Variables (Recommended)

Use `run_gaussian_elimination(...)` directly with explicitly defined
`variables`, `variables_old`, and `delt` symbols.

This avoids symbol-mapping ambiguities and is the most robust path for the current `acg_brns` version.

In [15]:
# Define 5-component, 3-reaction system
C1, C2, C3, C4, C5 = symbols('C1:6')
C1_old, C2_old, C3_old, C4_old, C5_old = symbols('C1_old C2_old C3_old C4_old C5_old')
k1, k2, k3 = symbols('k1 k2 k3', positive=True, real=True)
delt = symbols('delt', positive=True, real=True)

# Define reactions with rate constants
r1 = k1 * C1           # First-order reaction
r2 = k2 * C2 * C3      # Second-order reaction
r3 = k3 * C4           # First-order reaction

reactions_complex = [r1, r2, r3]

# Define net rates (stoichiometry)
equations_complex = [
    -r1 + r3,           # dA/dt: consumed by r1, produced by r3
    r1 - r2,            # dB/dt: produced by r1, consumed by r2
    -r2,                # dC/dt: consumed by r2
    r2 - r3,            # dD/dt: produced by r2, consumed by r3
    S(0)                # E: inert background species
]

print("Complex System: 5 components, 3 reactions with rate constants")
print(f"  Reactions:")
print(f"    r1 = k1*C1 (first-order)")
print(f"    r2 = k2*C2*C3 (second-order)")
print(f"    r3 = k3*C4 (first-order)")
print(f"\n  Inert component: C5")

ACG initialized: output=./generated_fortran_example, style=f77
Complex System: 5 components, 3 reactions with rate constants
  Reactions:
    r1 = k1*C1 (first-order)
    r2 = k2*C2*C3 (second-order)
    r3 = k3*C4 (first-order)

  Inert component: C5


### Run Pipeline and Compute Optimization Statistics

In [16]:
# Run robust pipeline with explicit symbols
optimized = run_gaussian_elimination(
    equations=equations_complex,
    variables=[C1, C2, C3, C4, C5],
    reactions=reactions_complex,
    variables_old=[C1_old, C2_old, C3_old, C4_old, C5_old],
    delt=delt,
    inert_components={4},  # C5 is inert
    verbose=False
)

# Compute optimization statistics locally
jac = optimized['jacobian']
ncompo = len(optimized['func'])
original_size = ncompo ** 2
nonzero_entries = sum(
    1 for i in range(ncompo) for j in range(ncompo) if jac[i, j] != 0
)
sparsity_ratio = 1 - (nonzero_entries / original_size) if original_size > 0 else 0
expected_speedup = original_size / nonzero_entries if nonzero_entries > 0 else float('inf')

print("\n" + "="*70)
print("OPTIMIZATION STATISTICS")
print("="*70)
print(f"\nOriginal Jacobian size: {original_size} entries")
print(f"Non-zero entries after optimization: {nonzero_entries}")
print(f"Pivot rows: {optimized['pivot_rows']}")
print(f"Conservation rows: {optimized['conservation_rows']}")
print(f"Sparsity ratio: {sparsity_ratio:.1%}")
print(f"Expected speedup (structural estimate): {expected_speedup:.2f}x")


OPTIMIZATION STATISTICS

Original Jacobian size: 25 entries
Non-zero entries after optimization: 13
Sparsity ratio: 48.0%
Pivot rows: [0, 1, 2]
